# 🚗 Parking Lot Ticketing System

- 10 slots: 4 small (bike only), 4 medium (bike, car), 2 large (bike, car, suv)
- ₹20 for the first hour, ₹10 for every hour after, rounded up


In [6]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import math
import re
from datetime import datetime

# ---------- Seed data ----------
SLOT_LAYOUT = [
    ("S1", "small"), ("S2", "small"), ("S3", "small"), ("S4", "small"),
    ("M1", "medium"), ("M2", "medium"), ("M3", "medium"), ("M4", "medium"),
    ("L1", "large"), ("L2", "large"),
]
FITS = {
    "small": ["bike"],
    "medium": ["bike", "car"],
    "large": ["bike", "car", "suv"],
}
SIZE_ORDER = ["small", "medium", "large"]

slots = {sid: {"size": size, "vehicle_no": None, "ticket_no": None} for sid, size in SLOT_LAYOUT}
tickets = {}          # ticket_no -> details
ticket_counter = 0
total_collected = 0

# ---------- Core logic ----------
def allocate_slot(vtype):
    """Smallest slot size the vehicle fits, scanning small -> medium -> large."""
    for size in SIZE_ORDER:
        if vtype not in FITS[size]:
            continue
        for sid, s in slots.items():
            if s["size"] == size and s["vehicle_no"] is None:
                return sid
    return None

def vehicle_already_parked(vehicle_no):
    return any(t["vehicle_no"] == vehicle_no and t["status"] == "active" for t in tickets.values())

def park(vehicle_no, vtype):
    global ticket_counter
    vehicle_no = vehicle_no.strip().upper()
    if not vehicle_no:
        return None, "Enter a vehicle number."
    if not (re.search(r"[A-Z]", vehicle_no) and re.search(r"[0-9]", vehicle_no)):
        return None, "Vehicle number must contain both letters and numbers."
    if vehicle_already_parked(vehicle_no):
        return None, f"{vehicle_no} is already parked."
    slot_id = allocate_slot(vtype)
    if slot_id is None:
        return None, f"Garage full: no free slot fits a {vtype} right now."
    ticket_counter += 1
    ticket_no = f"T{ticket_counter:03d}"
    entry_time = datetime.now()
    slots[slot_id]["vehicle_no"] = vehicle_no
    slots[slot_id]["ticket_no"] = ticket_no
    tickets[ticket_no] = {
        "vehicle_no": vehicle_no, "type": vtype, "slot_id": slot_id,
        "entry_time": entry_time, "status": "active",
        "exit_time": None, "amount": None,
    }
    return ticket_no, f"Parked. Slot {slot_id}, Ticket {ticket_no}, entry {entry_time.strftime('%I:%M %p')}"

def calc_charge(minutes):
    """Rs 20 for the first hour, Rs 10 for each hour after, rounding UP every part-hour."""
    hours = math.ceil(minutes / 60)
    if hours <= 1:
        return 20
    return 20 + (hours - 1) * 10

def unpark(ticket_no):
    global total_collected
    ticket_no = ticket_no.strip().upper()
    t = tickets.get(ticket_no)
    if t is None:
        return None, "Ticket number not found."
    if t["status"] != "active":
        return None, "This ticket has already been used."
    exit_time = datetime.now()
    minutes = (exit_time - t["entry_time"]).total_seconds() / 60
    amount = calc_charge(minutes)
    t["status"] = "closed"
    t["exit_time"] = exit_time
    t["amount"] = amount
    slot_id = t["slot_id"]
    slots[slot_id]["vehicle_no"] = None
    slots[slot_id]["ticket_no"] = None
    total_collected += amount
    return amount, f"Slot {slot_id} freed. Amount to pay: Rs {amount}"

def render_grid():
    boxes = ""
    for sid, s in slots.items():
        if s["vehicle_no"] is None:
            status, color = "FREE", "#e6ffe6"
        else:
            status, color = f'{s["vehicle_no"]}<br>({s["ticket_no"]})', "#ffe6e6"
        boxes += f'''
        <div style="background:{color}; border:1px solid #999; border-radius:6px;
                    padding:12px; width:110px; text-align:center; font-size:13px;">
          <div style="font-weight:bold">{sid}</div>
          <div>{s["size"]}</div>
          <div>{status}</div>
        </div>
        '''
    return f'''
    <div style="display:flex; flex-wrap:wrap; gap:10px;">{boxes}</div>
    <p><b>Total collected today: Rs {total_collected}</b></p>
    '''

# ---------- UI ----------
vehicle_input = widgets.Text(description="Vehicle No:", placeholder="KA01HH1234")
type_dropdown = widgets.Dropdown(options=["bike", "car", "suv"], description="Type:")
park_btn = widgets.Button(description="Park", button_style="success")

ticket_input = widgets.Text(description="Ticket No:", placeholder="T001")
unpark_btn = widgets.Button(description="Unpark", button_style="danger")

message_out = widgets.Output()
grid_out = widgets.Output()

def refresh_grid():
    with grid_out:
        clear_output()
        display(HTML(render_grid()))

def on_park_click(b):
    with message_out:
        clear_output()
        _, msg = park(vehicle_input.value, type_dropdown.value)
        print(msg)
    vehicle_input.value = ""
    refresh_grid()

def on_unpark_click(b):
    with message_out:
        clear_output()
        _, msg = unpark(ticket_input.value)
        print(msg)
    ticket_input.value = ""
    refresh_grid()

park_btn.on_click(on_park_click)
unpark_btn.on_click(on_unpark_click)

display(widgets.HTML("<h3>Park a vehicle</h3>"))
display(widgets.HBox([vehicle_input, type_dropdown, park_btn]))
display(widgets.HTML("<h3>Unpark a vehicle</h3>"))
display(widgets.HBox([ticket_input, unpark_btn]))
display(message_out)
display(widgets.HTML("<h3>Slots</h3>"))
refresh_grid()
display(grid_out)

HTML(value='<h3>Park a vehicle</h3>')

HTML(value='<h3>Unpark a vehicle</h3>')

Output()

HTML(value='<h3>Slots</h3>')

Output()